# 🕵️ Dia 26 — Investigação em redes sociais, parte 2: conteúdo arquivado e apagado

**Investigador Cyber 90 Dias** · Fase 2 · Rank Jutsu `C`

> Baseado no *Day 26* de [cyber-investigator-skills](https://github.com/thiefdroppers/cyber-investigator-skills/blob/main/curriculum/phase-2-osint-digital-footprint/day-26.md), © ThiefDroppers (MIT).
> Notebook, camada BR e Camada 8 por Deivison Lourenço ([@Ridd1kulusC0d3r](https://github.com/Ridd1kulusC0d3r)).

---

### 🎬 Onde paramos

O Dr. Hélio quer saber se o aviso "nunca enviamos SMS com link" já estava na página Segurança quando o golpe aconteceu. Ninguém lembra. Rafa: *"A Wayback lembra. Mas ela não vai te dizer o dia. Vai te dizer entre quais duas capturas mudou."*

Neste notebook você vai:

1. **Consultar a API CDX** (com rede, em `example.com`; sem rede, os dados do caso).
2. **Reimplementar `filter` e `collapse=digest`** e ver o que o colapso esconde.
3. **Jogar "em ou entre?"**.
4. **Calcular as janelas, fazer o diff e responder ao Dr. Hélio**.

🐍 Só biblioteca padrão do Python. Nada para instalar. Rode as células de cima para baixo (Shift+Enter). Tudo o que é do caso é **fictício** (domínios `.example`, IPs de documentação).

## 1. A API CDX

Primeiro, uma consulta real a `example.com` (domínio reservado para exemplos). Depois, os dados sintéticos do caso, no **mesmo formato** que a API devolve com `output=json`.

In [ ]:
import json, urllib.request, urllib.parse

def cdx(url, extra=""):
    """Consulta a API CDX da Wayback. Devolve lista de dicionários (sem a linha de cabeçalho)."""
    q = ("https://web.archive.org/cdx/search/cdx?url=" + urllib.parse.quote(url, safe="/*")
         + "&output=json" + extra)
    with urllib.request.urlopen(q, timeout=15) as r:
        linhas = json.load(r)
    if not linhas:
        return []
    cab = linhas[0]                                      # a primeira linha é o cabeçalho
    return [dict(zip(cab, l)) for l in linhas[1:]]

try:
    reais = cdx("example.com/", "&fl=timestamp,statuscode,digest&limit=5")
    print("Wayback respondeu. Primeiras capturas de example.com/:")
    for r in reais:
        print("  ", r)
except Exception as erro:
    print(f"(Sem acesso à Wayback agora: {type(erro).__name__}. Seguimos com os dados do caso.)")

In [ ]:
import base64, hashlib

# Conteúdo de cada versão (o que a cópia id_ devolveria) — idêntico a caso-continuo/dados/dia-26/copias-id_/
SEG_V1 = '<html><title>Seguranca</title><body><h1>Seguranca</h1><p>A Serra Azul nunca pede sua senha por telefone ou e-mail.</p></body></html>'
SEG_V2 = '<html><title>Seguranca</title><body><h1>Seguranca</h1><p>A Serra Azul nunca pede sua senha por telefone ou e-mail.</p><p>A Serra Azul nunca envia SMS com links para desbloqueio de chave Pix.</p></body></html>'
PIX_V1 = '<html><title>Regularizar</title><body><form><input name=agencia><input name=conta><input name=senha></form></body></html>'
PIX_V2 = '<html><title>Regularizar</title><body><form><input name=agencia><input name=conta><input name=senha><input name=cpf></form></body></html>'
PIX_V3 = '<html><title>Conta suspensa</title><body><p>Esta conta de hospedagem foi suspensa.</p></body></html>'
COPIAS = {"SEG_V1": SEG_V1, "SEG_V2": SEG_V2, "PIX_V1": PIX_V1, "PIX_V2": PIX_V2, "PIX_V3": PIX_V3}

def digest(conteudo):
    """Como a Wayback: SHA-1 do conteúdo, em base32. Mesmo conteúdo = mesmo digest."""
    return base64.b32encode(hashlib.sha1(conteudo.encode("utf-8")).digest()).decode("ascii")

def montar_cdx(original, capturas):
    """Recria a resposta CDX (lista de dicionários) a partir de (timestamp, status, nome_da_versão)."""
    linhas = []
    for ts, st, versao in capturas:
        c = COPIAS[versao] if versao else "<html>Moved</html>"
        linhas.append({"timestamp": ts, "original": original, "statuscode": str(st),
                       "digest": digest(c), "versao": versao})
    return linhas

SEG = montar_cdx("https://serraazul.example/seguranca", [
    ("20260310141502", 200, "SEG_V1"), ("20260610090011", 200, "SEG_V1"), ("20260815220540", 301, None),
    ("20260815220541", 200, "SEG_V1"), ("20260928173320", 200, "SEG_V1"),
    ("20261005180412", 200, "SEG_V2"), ("20261006091512", 200, "SEG_V2"), ("20261012110003", 200, "SEG_V2")])
PIX = montar_cdx("https://serraazul-pix.example/regularizar", [
    ("20261005101502", 200, "PIX_V1"), ("20261005152210", 200, "PIX_V1"), ("20261006081044", 200, "PIX_V1"),
    ("20261006213005", 200, "PIX_V2"), ("20261007120130", 200, "PIX_V2"), ("20261008184521", 200, "PIX_V2"),
    ("20261009120002", 200, "PIX_V3"), ("20261012090000", 200, "PIX_V3")])

print("Linhas brutas de /seguranca:")
for l in SEG:
    print(f"  {l['timestamp']}  {l['statuscode']}  {l['digest'][:8]}...")

## 2. `filter` e `collapse=digest`, à mão

`collapse=digest` guarda a **primeira** captura de cada sequência igual e joga fora o resto, inclusive a **última**, que é justamente a que você precisa para a janela. Por isso a janela se lê na lista **não colapsada**.

In [ ]:
def filtrar_200(linhas):
    """filter=statuscode:200 — descarta redirecionamentos e erros."""
    return [l for l in linhas if l["statuscode"] == "200"]

def colapsar(linhas):
    """collapse=digest — compara cada linha só com a ANTERIOR (adjacente)."""
    saida, anterior = [], None
    for l in linhas:
        if l["digest"] != anterior:
            saida.append(l)
        anterior = l["digest"]
    return saida

def versoes_com_janela(linhas):
    """Para cada versão: 1ª captura, última captura idêntica e a janela da mudança seguinte."""
    ok = filtrar_200(linhas)
    blocos = []                                            # [(digest, [timestamps...])]
    for l in ok:
        if blocos and blocos[-1][0] == l["digest"]:
            blocos[-1][1].append(l["timestamp"])
        else:
            blocos.append((l["digest"], [l["timestamp"]], l["versao"]))
    return blocos

def fmt(ts):
    return f"{ts[0:4]}-{ts[4:6]}-{ts[6:8]} {ts[8:10]}:{ts[10:12]} UTC"

for nome, linhas in (("/seguranca", SEG), ("/regularizar (site falso)", PIX)):
    print(f"\n{nome}: {len(linhas)} capturas brutas → {len(colapsar(filtrar_200(linhas)))} versões distintas")
    blocos = versoes_com_janela(linhas)
    for i, (dg, tss, versao) in enumerate(blocos, start=1):
        print(f"  v{i} ({versao}) {dg[:6]}  primeira {fmt(tss[0])}  última idêntica {fmt(tss[-1])}")
    for (d1, t1, _v1), (d2, t2, _v2) in zip(blocos, blocos[1:]):
        print(f"  ↳ mudança ENTRE {fmt(t1[-1])} e {fmt(t2[0])}")

## 3. 🎮 Em ou entre?

1 = frase honesta · 2 = frase que afirma mais do que o arquivo sustenta

In [ ]:
QUIZ = [
    ("'O aviso sobre SMS foi publicado em 5/out/2026.'", "2",
     "A captura de 5/out 18h é a PRIMEIRA com o aviso; ele pode ter entrado em qualquer momento desde 28/09."),
    ("'O aviso foi acrescentado entre 2026-09-28 17:33 UTC e 2026-10-05 18:04 UTC.'", "1",
     "Última captura sem + primeira com. É o que a evidência sustenta."),
    ("'O aviso só apareceu depois do golpe.'", "2",
     "A janela começa antes dos disparos (5/out 10:12 UTC). O arquivo não permite essa conclusão."),
    ("'O campo de CPF entrou no site falso entre 06/out 08:10 UTC e 06/out 21:30 UTC.'", "1",
     "Janela entre duas capturas. (A sua captura do Dia 19, às 09:13 sem CPF, estreita ainda mais.)"),
    ("'A página mudou no dia 15/08/2026, quando houve o 301.'", "2",
     "O 301 é um redirecionamento: a URL mudou de lugar. O digest do conteúdo continuou o mesmo."),
]
MODO_JOGO = True
acertos = 0
for frase, certo, porque in QUIZ:
    print(f"\n• {frase}")
    if MODO_JOGO:
        r = ""
        while r not in ("1", "2"):
            r = input("   1 honesta / 2 afirma demais: ").strip()
        acertos += (r == certo)
        print("   ✅" if r == certo else "   ❌", end="")
    print(f"   {porque}")
if MODO_JOGO:
    print(f"\n🏁 {acertos}/{len(QUIZ)}")

## 4. Diff, hash, densidade e a resposta ao Dr. Hélio

In [ ]:
import difflib, re
from pathlib import Path

Path("captures").mkdir(exist_ok=True)
for nome, conteudo in COPIAS.items():                  # salva as cópias id_ e calcula o hash (Dia 19)
    p = Path("captures") / f"{nome.lower()}.html"
    p.write_text(conteudo, encoding="utf-8")
    print(f"{hashlib.sha256(p.read_bytes()).hexdigest()}  {p}")

def texto(html):
    """O mesmo que o sed do original: tira as tags, sobra o texto."""
    return [t.strip() for t in re.sub(r"<[^>]*>", "\n", html).splitlines() if t.strip()]

for a, b in (("SEG_V1", "SEG_V2"), ("PIX_V1", "PIX_V2"), ("PIX_V2", "PIX_V3")):
    print(f"\n--- diff {a} → {b}")
    for linha in difflib.unified_diff(texto(COPIAS[a]), texto(COPIAS[b]), a, b, lineterm="", n=0):
        if linha.startswith(("+", "-")) and not linha.startswith(("+++", "---")):
            print("  ", linha)
# O formulário mudou só num atributo: o diff de texto não pega tags. Compare o HTML cru:
print("\nCampos do formulário v1:", re.findall(r"name=(\w+)", PIX_V1))
print("Campos do formulário v2:", re.findall(r"name=(\w+)", PIX_V2), "← o sed do original não mostraria isso")

In [ ]:
from collections import Counter
# Densidade (Passo 3): carimbos sintéticos de capturas sob serraazul.example/*
POR_MES = {"202601": 3, "202602": 2, "202603": 5, "202604": 2, "202605": 3, "202606": 4,
           "202607": 2, "202608": 3, "202609": 4, "202610": 61}
carimbos = [f"{m}{(i % 28) + 1:02d}{(i * 7) % 24:02d}{(i * 13) % 60:02d}00" for m, n in POR_MES.items() for i in range(n)]
contagem = Counter(c[:6] for c in carimbos)            # = cut -c1-6 | sort | uniq -c
maior = max(contagem.values())
for mes in sorted(contagem):
    barra = "█" * max(1, round(contagem[mes] * 40 / maior))
    print(f"{mes[:4]}-{mes[4:]}  {contagem[mes]:>3}  {barra}")
print("\nPico de out/2026: lembre que a PRÓPRIA investigação (Save Page Now no Dia 19) pede capturas.")

In [ ]:
#@title Resposta ao Dr. Hélio { display-mode: "form" }
ultima_sem_aviso = "2026-09-28 17:33 UTC"  #@param {type:"string"}
primeira_com_aviso = "2026-10-05 18:04 UTC"  #@param {type:"string"}
fonte_mais_precisa = "log de publicação do CMS da cooperativa (pedir à TI via Marta)"  #@param {type:"string"}

resposta = (f"O aviso 'nunca enviamos SMS com link' foi acrescentado à página Segurança ENTRE {ultima_sem_aviso} "
            f"(última captura da Wayback sem o aviso) e {primeira_com_aviso} (primeira captura com ele). "
            f"O arquivo não permite dizer se foi antes ou depois dos disparos de 05/10 às 07h12 BRT. "
            f"Fonte que daria mais precisão: {fonte_mais_precisa}. "
            f"Cópias id_ com SHA-256 no registro de coleta.")
# Trava: a resposta não pode afirmar um dia único.
assert "ENTRE" in resposta and " e " in resposta, "A resposta precisa ser uma janela."
print(resposta)
Path("notas").mkdir(exist_ok=True)
Path("notas/resposta-dr-helio.md").write_text("# Resposta ao Dr. Hélio\n\n" + resposta + "\n", encoding="utf-8")
print("\n💾 notas/resposta-dr-helio.md")

### 🧠 Camada 8

As duas partes querem um dia que as favoreça; a memória da Marta oscila porque é **reconstrução** (viés retrospectivo). E o analista sente a **aversão à ambiguidade**: "entre 28/09 e 05/10" parece pior que "dia 3". Não é. É a resposta que a evidência sustenta.

## ✅ Antes de fechar o Dia 26

- [ ] Você sabe por que a janela se lê na lista **não colapsada**
- [ ] Cópias `id_` salvas com hash
- [ ] Tabela de versões das duas URLs, com as janelas
- [ ] Gráfico de densidade e a frase sobre o pico de outubro
- [ ] `notas/resposta-dr-helio.md` sem nenhum "dia único"

**Amanhã (Dia 27):** metadados de imagem. Os prints que os cooperados mandaram têm data, aparelho e, às vezes, lugar. Ou não têm.

---
<sub>Baseado em cyber-investigator-skills por ThiefDroppers (MIT) · Investigador Cyber 90 Dias por Deivison Lourenço</sub>